# BSCP sack-train-ml — Compile YOLOv11s → HEF (Colab · DFC ClientRunner)

> Loom Oracle (AI) · 2026-06-23 · คู่กับ `train_run.ipynb` (อันนั้นเทรน, อันนี้ compile)
> pipeline (ตรงกับ DFC API): **`.onnx → parse(.har) → optimize/quantize+calibrate(.har) → compile(.hef)`**

## ▶ เริ่ม: Runtime → Change runtime type → **High-RAM** (GPU ถ้ามี ช่วย optimize เร็วขึ้น) → รันทีละ section

---
### ⚠️ เตือน 1 — เวอร์ชันต้องโหลดบน edge ได้
edge รัน **HailoRT 4.20.0** → HEF ต้องโหลดบน 4.20.0 ได้
- ชัวร์สุด: **DFC 3.30.0 + model-zoo 2.14.0** (Hailo Suite 2025-01)
- ถ้าใช้ DFC ใหม่กว่า (3.33.x) → **ต้องทดสอบโหลดบน Pi จริง** (section 11) ถ้า error `invalid compiled format` ค่อยถอยมา 3.30

### ✅ เตือน 2 — โมเดล custom 2 คลาส (verified โดย Loom)
best.onnx นี้ถูกตรวจแล้ว: detect head `model.23`, cls conv = **2 channels** (Person, Sack), box conv = 64 (4×DFL16).
6 end-nodes + input name (`images`) ใส่ไว้ใน config ด้านล่างจาก onnx จริง — ไม่ต้องเดา


## 0) Config — แก้ค่าตรงนี้ที่เดียว


In [ ]:
# ===== config =====
HW_ARCH      = 'hailo8l'
NET_NAME     = 'yolov11s_sack'
NUM_CLASSES  = 2
CLASS_NAMES  = ['Person', 'Sack']
INPUT_SIZE   = 640
START_NODE   = 'images'                 # verified จาก onnx

# 6 end-nodes (verified จาก best.onnx) — จุดตัดก่อน DFL+sigmoid ให้ NMS บนชิปทำต่อ
END_NODES = [
    '/model.23/cv2.0/cv2.0.2/Conv', '/model.23/cv3.0/cv3.0.2/Conv',   # stride 8  (box64, cls2)
    '/model.23/cv2.1/cv2.1.2/Conv', '/model.23/cv3.1/cv3.1.2/Conv',   # stride 16
    '/model.23/cv2.2/cv2.2.2/Conv', '/model.23/cv3.2/cv3.2.2/Conv',   # stride 32
]

# NMS contract (จาก edge hailo_backend.py analysis)
NMS_SCORES_TH = 0.20    # <=0.25 กัน flagging band [0.35,0.70) หาย
NMS_IOU_TH    = 0.70    # สูงไว้ กันสองกระสอบติดกันโดน merge เป็นอันเดียว
MAX_PER_CLASS = 50      # = --max_det ของ edge
REG_LENGTH    = 16      # DFL (box conv 64 = 4*16)

# paths
ONNX_PATH = '/content/best.onnx'        # อัปขึ้น Colab (section 1)
CALIB_DIR = '/content/calib'            # >=256 รูปจริงจาก edge
WORK      = '/content/work'
OUT_HEF   = f'{WORK}/{NET_NAME}.hef'
CALIB_N   = 512                         # 256-1024; มากขึ้น = quantize นิ่งขึ้น แต่ช้าลง

SOURCE_COMMIT = 'cb0c3ab'   # sack-train-ml sha ตอน export onnx (ใส่ของจริง)

import os; os.makedirs(WORK, exist_ok=True)
print('config ok:', NET_NAME, NUM_CLASSES, 'classes ->', HW_ARCH, '| end_nodes:', len(END_NODES))


## 1) อัปไฟล์ขึ้น Colab
ต้องมี 4 อย่าง:
1. **DFC whl** `hailo_dataflow_compiler-3.x-...whl` (Hailo Developer Zone, gated)
2. **model-zoo whl** `hailo_model_zoo-*.whl` (optional — ClientRunner ไม่ต้องใช้ แต่ติดไว้สำหรับ eval)
3. **best.onnx** → `/content/best.onnx`
4. **calib/** ≥256 รูปจริงจาก edge → `/content/calib/`  (clean frames ที่เราเตรียมไว้ใช้ได้)

แนะนำเก็บใน Google Drive แล้ว mount (ไม่ต้องอัปซ้ำทุก session)


In [ ]:
# ทางเลือก A: mount Drive (แนะนำ)
from google.colab import drive
drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/hailo/'*.whl /content/
# !cp '/content/drive/MyDrive/hailo/best.onnx' /content/best.onnx
# !cp -r '/content/drive/MyDrive/hailo/calib' /content/calib


In [ ]:
# ทางเลือก B: อัปไฟล์เล็กตรง ๆ
# from google.colab import files; files.upload()
import glob, os
print('DFC whl :', glob.glob('/content/hailo_dataflow_compiler*.whl'))
print('MZ  whl :', glob.glob('/content/hailo_model_zoo*.whl'))
print('onnx    :', os.path.exists(ONNX_PATH))
print('calib n :', len(glob.glob(f'{CALIB_DIR}/*.jpg')+glob.glob(f'{CALIB_DIR}/*.png')+glob.glob(f'{CALIB_DIR}/*.jpeg')))


## 2) Runtime check (High-RAM)
compile กิน RAM 12–32GB, ใช้ CPU เป็นหลัก (GPU ช่วย optimize)


In [ ]:
import subprocess, sys
print(subprocess.run(['free','-h'],capture_output=True,text=True).stdout)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout or 'no GPU (CPU-only, ช้าแต่ได้)')
print('python', sys.version.split()[0], '(DFC ต้องการ 3.10/3.11)')


## 3) System deps + pins
Colab cuDNN 9 ทำ DFC พัง — pin numpy/scipy ที่ DFC ต้องการ


In [ ]:
!sudo apt-get -qq update
!sudo apt-get -qq install -y python3-dev python3-tk libfuse2 graphviz libgraphviz-dev >/dev/null 2>&1
print('apt deps ok')


## 4) venv + ติดตั้ง DFC
DFC อยู่ใน virtualenv แยก (กัน dep ชนกับ kernel). ClientRunner เรียกผ่าน `!hailo_venv/bin/python script.py`


In [ ]:
!pip -q install virtualenv
!virtualenv -p python3 /content/hailo_venv >/dev/null
VENV='/content/hailo_venv/bin'
!{VENV}/pip install --upgrade pip wheel >/dev/null
!{VENV}/pip install numpy==1.23.3 scipy==1.10.1 pillow >/dev/null
!{VENV}/pip install /content/hailo_dataflow_compiler*.whl
# model-zoo optional (eval เท่านั้น):
# !{VENV}/pip install /content/hailo_model_zoo*.whl


In [ ]:
# verify DFC พร้อม + เวอร์ชัน
!/content/hailo_venv/bin/python -c "from hailo_sdk_client import ClientRunner; import hailo_sdk_client as c; print('hailo_sdk_client', getattr(c,'__version__','?'))"


## 5) Calibration set
คุณภาพ calib = ตัวชี้เป็นชี้ตายของ recall หลัง quantize (โดยเฉพาะคลาสน้อย = Person).
ใช้ **รูปจริงจาก edge** ครอบคลุม กลางวัน/กลางคืน/ฝุ่น/บัง/มุมกล้องจริง ≥256 รูป.
compile script จะ **letterbox 640 (pad 114) แบบเดียวกับ edge** แล้วป้อนเป็น 0–255 (normalization layer หาร 255 บนชิป)


In [ ]:
# manifest + dataset hash (ไปใส่ meta.yaml)
import glob, hashlib, csv
imgs = sorted(glob.glob(f'{CALIB_DIR}/*.jpg')+glob.glob(f'{CALIB_DIR}/*.jpeg')+glob.glob(f'{CALIB_DIR}/*.png'))
assert len(imgs) >= 256, f'ต้องการ >=256 รูป มี {len(imgs)} — เพิ่ม calib ก่อน'
h = hashlib.sha256()
with open(f'{WORK}/calibration_manifest.csv','w',newline='') as f:
    w = csv.writer(f)
    for p in imgs:
        d = hashlib.sha256(open(p,'rb').read()).hexdigest(); h.update(d.encode()); w.writerow([p,d])
DATASET_HASH = h.hexdigest()
print('calib:', len(imgs), '| dataset_hash:', DATASET_HASH[:16], '...')


## 6) เขียน compile script (ClientRunner)
`parse → save_har → load_model_script(normalization+nms) → optimize(calib) → save_har → compile → hef`.
รันใน hailo_venv (DFC isolated). script จะ **print ชื่อ HAR output layers หลัง parse** — ใช้ตรวจ/แมพ nms bbox_decoders


In [ ]:
%%writefile /content/compile_clientrunner.py
#!/usr/bin/env python
"""ONNX -> HAR (parse) -> quantize+calibrate (HAR) -> HEF, via DFC ClientRunner.
Custom YOLOv11s 2-class {Person,Sack}, Hailo-8L. Matches edge contract:
uint8 640 letterbox input (on-chip /255), on-chip NMS, 2 classes.
"""
import argparse, glob, json, os
import numpy as np
from PIL import Image

def letterbox(im, size, color=(114,114,114)):
    w,h = im.size; r = min(size/w, size/h)
    nw,nh = round(w*r), round(h*r)
    im = im.resize((nw,nh), Image.BILINEAR)
    cv = Image.new('RGB',(size,size),color)
    cv.paste(im, ((size-nw)//2,(size-nh)//2)); return cv

def load_calib(d, n, size):
    fs = sorted(glob.glob(d+'/*.jpg')+glob.glob(d+'/*.jpeg')+glob.glob(d+'/*.png'))[:n]
    if not fs: raise SystemExit('no calib images in '+d)
    a = np.zeros((len(fs),size,size,3), np.float32)
    for i,f in enumerate(fs): a[i]=np.asarray(letterbox(Image.open(f).convert('RGB'),size),np.float32)
    print(f'calib {len(fs)} imgs shape={a.shape} range=[{a.min():.0f},{a.max():.0f}]'); return a

def main():
    ap=argparse.ArgumentParser()
    for k in ['onnx','calib','out','work']: ap.add_argument('--'+k, required=True)
    ap.add_argument('--hw',default='hailo8l'); ap.add_argument('--net',default='yolov11s_sack')
    ap.add_argument('--classes',type=int,default=2); ap.add_argument('--size',type=int,default=640)
    ap.add_argument('--calib-n',type=int,default=512)
    ap.add_argument('--scores-th',type=float,default=0.20); ap.add_argument('--iou-th',type=float,default=0.70)
    ap.add_argument('--max-per-class',type=int,default=50); ap.add_argument('--reg-len',type=int,default=16)
    a=ap.parse_args(); os.makedirs(a.work,exist_ok=True)
    from hailo_sdk_client import ClientRunner

    START='images'
    END=['/model.23/cv2.0/cv2.0.2/Conv','/model.23/cv3.0/cv3.0.2/Conv',
         '/model.23/cv2.1/cv2.1.2/Conv','/model.23/cv3.1/cv3.1.2/Conv',
         '/model.23/cv2.2/cv2.2.2/Conv','/model.23/cv3.2/cv3.2.2/Conv']

    # ---- 1) PARSE: onnx -> HAR ----
    r = ClientRunner(hw_arch=a.hw)
    r.translate_onnx_model(a.onnx, a.net, start_node_names=[START], end_node_names=END,
                           net_input_shapes={START:[1,3,a.size,a.size]})
    parsed=f'{a.work}/{a.net}_parsed.har'; r.save_har(parsed); print('PARSED ->',parsed)

    # print HAR output layers (ใช้ตรวจ/แมพ nms bbox_decoders)
    try:
        hn = r.get_hn() if hasattr(r,'get_hn') else None
        import json as _j; hd = _j.loads(hn) if isinstance(hn,str) else hn
        outs = [n for n,l in (hd or {}).get('layers',{}).items() if l.get('type')=='output_layer'] if hd else []
        print('HAR output layers:', outs)
    except Exception as e: print('(layer introspection skipped:', e, ')')

    # ---- 2) model script: normalization (edge ป้อน uint8 ดิบ) + on-chip NMS 2-class ----
    nms = {'nms_scores_th':a.scores_th,'nms_iou_th':a.iou_th,'image_dims':[a.size,a.size],
           'max_proposals_per_class':a.max_per_class,'classes':a.classes,
           'regression_length':a.reg_len,'background_removal':False}
    nms_json=f'{a.work}/nms_config.json'; json.dump(nms,open(nms_json,'w'),indent=2)
    alls=(f'normalization1 = normalization([0.0,0.0,0.0],[255.0,255.0,255.0])\n'
          f'nms_postprocess("{nms_json}", meta_arch=yolov8, engine=cpu)\n')
    print('--- alls ---\n'+alls)
    r.load_model_script(alls)

    # ---- 3) QUANTIZE + CALIBRATE: -> quantized HAR ----
    calib = load_calib(a.calib, a.calib_n, a.size)
    r.optimize(calib)
    quant=f'{a.work}/{a.net}_quantized.har'; r.save_har(quant); print('QUANTIZED ->',quant)

    # ---- 4) COMPILE -> HEF ----
    hef = r.compile()
    open(a.out,'wb').write(hef); print('HEF ->',a.out,os.path.getsize(a.out),'bytes')

if __name__=='__main__': main()


## 7) Run: parse → quantize → compile
⚠️ ถ้า error ที่ `nms_postprocess` (syntax ขึ้นกับเวอร์ชัน DFC) → ดู `HAR output layers` ที่ print ออกมา แล้วเติม `bbox_decoders` ใน nms_config ตาม DFC docs ของเวอร์ชันเธอ


In [ ]:
VENV='/content/hailo_venv/bin'
!{VENV}/python /content/compile_clientrunner.py \
  --onnx {ONNX_PATH} --calib {CALIB_DIR} --out {OUT_HEF} --work {WORK} \
  --hw {HW_ARCH} --net {NET_NAME} --classes {NUM_CLASSES} --size {INPUT_SIZE} \
  --calib-n {CALIB_N} --scores-th {NMS_SCORES_TH} --iou-th {NMS_IOU_TH} \
  --max-per-class {MAX_PER_CLASS} --reg-len {REG_LENGTH}
import glob; print('HEF:', glob.glob(f'{WORK}/*.hef'))


## 8) ✅ Verify HEF ตรง edge contract (จับ silent failure ก่อน deploy)
input ต้อง **UINT8 640×640×3**, output ต้องเป็น **NMS**


In [ ]:
from hailo_platform import HEF
import glob
hef = HEF(glob.glob(f'{WORK}/*.hef')[0])
i = hef.get_input_vstream_infos()[0]; outs = hef.get_output_vstream_infos()
print('input  :', i.name, i.shape, i.format.type)
print('outputs:', [(o.name, str(o.format.order)) for o in outs])
assert 'UINT8' in str(i.format.type), '❌ input ไม่ใช่ uint8 → edge ป้อน uint8 จะ garbage (เช็ค normalization)'
assert list(i.shape[:2]) == [INPUT_SIZE,INPUT_SIZE], f'❌ input shape {i.shape}'
assert any('NMS' in str(o.format.order).upper() for o in outs), '❌ output ไม่ใช่ NMS (เช็ค nms_postprocess)'
print('✅ contract เบื้องต้นผ่าน (uint8 + shape + NMS). คลาส=2 ยืนยันตอน eval/บน Pi)')


## 9) Eval ด้วย emulator (วัด int8 mAP ไม่ต้องมีชิป) — optional
เทียบ fp32 vs int8 ตาม gate: mAP50 drop ≤3%, recall drop ≤5%. ต้องมี eval set (รูป+label) + model-zoo whl


In [ ]:
# ต้องติด model-zoo whl ก่อน (cell 4) + มี eval set
# VENV='/content/hailo_venv/bin'
# !{VENV}/hailomz eval --hw-arch {HW_ARCH} --target emulator \
#    --har {WORK}/{NET_NAME}_quantized.har {NET_NAME}   # --data-path /content/eval
print('eval optional — หรือใช้ event-GT ของ Loom วัด count-error บน vid1/2/3 (ตรงงานจริงกว่า)')


## 10) meta.yaml + download HEF
runtime ฝั่ง Pi ปฏิเสธถ้า sha256 ไม่ตรง


In [ ]:
import hashlib, glob, yaml
hef_path = glob.glob(f'{WORK}/*.hef')[0]
sha = hashlib.sha256(open(hef_path,'rb').read()).hexdigest()
meta = {
    'model': NET_NAME,
    'version': 'colab-2026-06-23',
    'source_commit': SOURCE_COMMIT,
    'dataset_hash': DATASET_HASH,
    'hailort_version': '4.20.0',
    'input_shape': [INPUT_SIZE, INPUT_SIZE, 3],
    'class_names': CLASS_NAMES,
    'end_node_names': END_NODES,
    'quantization': {'mode': 'int8', 'calibration_size': CALIB_N, 'per_channel': True},
    'nms': {'scores_th': NMS_SCORES_TH, 'iou_th': NMS_IOU_TH, 'max_per_class': MAX_PER_CLASS, 'classes': NUM_CLASSES},
    'accuracy': {'fp32': {'mAP50': None, 'recall': None}, 'int8': {'mAP50': None, 'recall': None}},  # เติมจาก eval
    'gates_passed': False,
    'sha256': sha,
}
mp = hef_path + '.meta.yaml'
with open(mp,'w') as f: yaml.safe_dump(meta, f, sort_keys=False, allow_unicode=True)
print(open(mp).read())
from google.colab import files; files.download(hef_path); files.download(mp)


## 11) ✅ ทดสอบบน Pi (สำคัญสุด — พิสูจน์เวอร์ชันตรง)
```bash
scp yolov11s_sack.hef yolov11s_sack.hef.meta.yaml edge-rpi:~/
hailortcli fw-control identify     # ยืนยัน HAILO8L + HailoRT 4.20.0
hailortcli run yolov11s_sack.hef   # โหลด+รันได้ = เวอร์ชันเข้ากันได้
```
error `invalid compiled format` = DFC/HailoRT ไม่ match → ถอยมา DFC 3.30.0 + model-zoo 2.14.0 แล้ว compile ใหม่

---
_Loom Oracle (AI) — end-nodes/input/2-class verified จาก best.onnx จริง (2026-06-23). จุดที่อาจต้องปรับตามเวอร์ชัน DFC: `nms_postprocess` syntax + bbox_decoders (ดู HAR output layers ที่ print ใน section 7)._
